# ***Task 1.3 — What the Paper Claims to Improve***

**Paper:** Pegasos: Primal Estimated sub-GrAdient SOlver for SVM  

## *Main Baseline*

The primary baseline identified and compared against in the paper is **SVM-Light** (Joachims, 1999), a widely-used dual decomposition-based SVM solver. The paper also prominently compares against **SVM-Perf** (Joachims, 2006), which is an optimised large-scale variant of SVM-Light designed specifically for linear kernels using a cutting-plane method.

## *Limitation of the Baseline Identified by the Paper*

SVM-Light and SVM-Perf both operate in the **dual domain**: they optimize over the Lagrange multipliers {α_i}. The key limitation the paper identifies is that dual methods require a number of iterations that grows at least **linearly with the training set size m**. SVM-Light requires O(m²) iterations to converge in the worst case (each iteration solving a QP over a working set), and even SVM-Perf, though faster, requires O(1/(λε)) iterations where each iteration processes the **entire** training set. As a result, both solvers scale poorly — their runtime to reach ε-accuracy is O(md/(λε)) at minimum — which becomes prohibitive when m reaches millions of examples, as in modern text classification.

## *How Pegasos Overcomes This Limitation*

Pegasos overcomes the dependence on m by working in the **primal domain** and using **stochastic sub-gradient descent** with a carefully chosen decaying learning rate `η_t = 1/(λt)`. Because it samples only a small mini-batch of k examples per iteration (often k=1), the per-iteration cost is O(kd) — independent of m. The total number of iterations to reach ε-accuracy is O(1/(λε)), also independent of m. Thus the overall runtime is O(d/(λε)), which is **asymptotically independent of the training set size**, unlike all dual methods.

## *One Condition Under Which Pegasos Would NOT Outperform the Baseline*

Pegasos would fail to outperform SVM-Perf (or even plain SVM-Light) in the regime of **very small datasets with a strong need for high precision in the dual solution**. Specifically:

- When m is small (e.g., m ≤ 1,000 samples), SVM-Light's dual solver can find the globally optimal support vectors with machine precision in a small number of solver steps. In contrast, Pegasos is a first-order stochastic method and converges at a rate of `O(1/T)` — reaching, say, 0.1% primal gap requires `T = O(1/(λ × 0.001))` iterations, which may be more work than the non-stochastic dual on a small dataset.
- Furthermore, the paper's speed advantage is demonstrated empirically on large datasets like Reuters-21578 and covtype (tens of thousands to millions of examples). The authors' own experimental results (Section 6) show that SVM-Perf is competitive with Pegasos when the dataset is small or when very high precision is required.
- My own reasoned inference is that on **highly imbalanced datasets** (one class is 1% of the data), random mini-batch sampling will rarely encounter the minority-class examples, making the stochastic gradient biased — in this case, a deterministic dual method that can explicitly focus on hard examples (support vectors) will converge more reliably to a high-quality boundary than a uniformly-sampling stochastic method like Pegasos.
